In [1]:
import pandas as pd
import numpy as np

# load the two .txt files
df_anti_test_type1 = pd.read_csv('anti_stereotyped_type1.txt.test',header=None, sep='\t')
df_anti_test_type2 = pd.read_csv('anti_stereotyped_type2.txt.test',header=None, sep='\t')
df_anti_train_type1 = pd.read_csv('anti_stereotyped_type1.txt.dev',header=None, sep='\t')
df_anti_train_type2 = pd.read_csv('anti_stereotyped_type2.txt.dev',header=None, sep='\t')
df_stereotyped_train_type1 = pd.read_csv('pro_stereotyped_type1.txt.dev',header=None, sep='\t')
df_stereotyped_train_type2 = pd.read_csv('pro_stereotyped_type2.txt.dev',header=None, sep='\t')
df_stereotyped_test_type1 = pd.read_csv('pro_stereotyped_type1.txt.test',header=None, sep='\t')
df_stereotyped_test_type2 = pd.read_csv('pro_stereotyped_type2.txt.test',header=None, sep='\t')

# define the first column as the text column
dfs = [df_anti_test_type1, df_anti_test_type2, df_anti_train_type1, df_anti_train_type2,
         df_stereotyped_train_type1, df_stereotyped_train_type2, df_stereotyped_test_type1, df_stereotyped_test_type2]

# save for the list if it is train/test
split_list = ['test', 'test', 'train', 'train',
               'train', 'train', 'test', 'test']
type_list = [1, 2, 1, 2, 1, 2, 1, 2]

stereotype_list = ['anti_stereotype', 'anti_stereotype', 'anti_stereotype', 'anti_stereotype',
              'stereotype', 'stereotype', 'stereotype', 'stereotype']

# loop over the dataframes and add the split and type columns
i = 0
for df in dfs:
    # change the column names to text
    df.columns = ['text']
    
    # clean up the text column
    df['text'] = df['text'].str.replace(r'^\d+\s+', '', regex=True)
    
    # add the split column
    df['split'] = split_list[i]
    
    # add the type column
    df['type_text'] = type_list[i]
    
    # add the type column
    df['anti_stereotype'] = 1 if stereotype_list[i] == 'anti_stereotype' else 0

    # add
    i += 1
    

# combine the two dataframes
df_combined = pd.concat(dfs, ignore_index=True)
df_combined


,text,split,type_text,anti_stereotype
0,The janitor reprimanded [the accountant] becau...,test,1,1
1,[The janitor] reprimanded the accountant becau...,test,1,1
2,[The chief] hired the assistant because [she] ...,test,1,1
3,The chief hired [the assistant] because [he] h...,test,1,1
4,[The carpenter] met with the teacher so that [...,test,1,1
...,...,...,...,...
3163,The driver was annoyed with [the assistant] an...,test,2,0
3164,The cleaner waited for [the chief] and talked ...,test,2,0
3165,The chief waited for [the cleaner] and asked [...,test,2,0
3166,The teacher befriended [the janitor] and told ...,test,2,0


In [2]:
# loop over each piece of text
for i in range(len(df_combined)):
    
    # get the text
    text = df_combined.iloc[i]['text']
    
    # get the profession, pronoun
    profession = text.split("[")[1].split("]")[0]
    pronoun = text.split("[")[2].split("]")[0]
    
    # get the clean sentence
    clean_sentence = text.replace(f"[{profession}]", profession).replace(f"[{pronoun}]", pronoun).strip()
    
    # use the correct profession, to be passed into a tokenizer
    profession_for_tokenizer = " ".join(profession.split(" ")[1:])
    
    # if it is nan, show the profession
    if profession_for_tokenizer == '':
        profession_for_tokenizer = profession.strip()
        print(f"Profession for tokenizer is empty for index {i}. Using:{profession_for_tokenizer}")
    
    # create the prompt
    prompt = clean_sentence + f" '{pronoun.capitalize()}' refers to the"
    
    # add all to the dataframe
    df_combined.at[i, 'profession'] = profession
    df_combined.at[i, 'pronoun'] = pronoun
    df_combined.at[i, 'clean_sentence'] = clean_sentence
    df_combined.at[i, 'prompt'] = prompt
    df_combined.at[i, 'profession_for_tokenizer'] = profession_for_tokenizer
    
# save the dataframe to a .csv file
df_combined.to_csv('winobias.csv', index=False)    

 # based on pronoun, map gender
he_she_map = {
    'he':'m', 'him':'m', 'his':'m',
    'she':'f', 'her':'f', 'hers':'f'
}
df_combined['gender'] = df_combined['pronoun'].map(he_she_map)

# turn it binary
df_combined['gender_binary']= df_combined['gender'].map({'m': 1, 'f': 0})

# unique values for profession_for_tokenizer
unique_professions = df_combined['profession_for_tokenizer'].unique()


# only leave type_text==1
df_combined = df_combined[df_combined['type_text'] == 1]


# save the dataframe to a .csv file
#df_combined.to_csv('winobias.csv', index=False)

Profession for tokenizer is empty for index 953. Using:housekeeper


In [3]:
# print the number of datapoints per profession
for profession in unique_professions:
    count = len(df_combined[df_combined['profession_for_tokenizer'] == profession])
    print(f"Profession: {profession}, Count: {count}")

Profession: accountant, Count: 42
Profession: janitor, Count: 42
Profession: chief, Count: 40
Profession: assistant, Count: 38
Profession: carpenter, Count: 40
Profession: teacher, Count: 40
Profession: lawyer, Count: 40
Profession: laborer, Count: 40
Profession: designer, Count: 40
Profession: cook, Count: 40
Profession: clerk, Count: 40
Profession: analyst, Count: 38
Profession: cashier, Count: 40
Profession: guard, Count: 38
Profession: writer, Count: 38
Profession: housekeeper, Count: 38
Profession: CEO, Count: 36
Profession: hairdresser, Count: 40
Profession: cleaner, Count: 40
Profession: counselor, Count: 40
Profession: developer, Count: 40
Profession: manager, Count: 38
Profession: mover, Count: 40
Profession: editor, Count: 40
Profession: farmer, Count: 40
Profession: attendant, Count: 40
Profession: baker, Count: 40
Profession: receptionist, Count: 40
Profession: construction worker, Count: 40
Profession: driver, Count: 40
Profession: auditor, Count: 38
Profession: salesperso